In [ ]:
!pip install llama-index llama-hub pypdf llamaapi

In [ ]:
# setup OpenAI
import openai

openai.api_key = ""

In [ ]:
import nest_asyncio

nest_asyncio.apply()

In [ ]:
from IPython.display import HTML, display

def set_css():
  display(HTML('''
  <style>
    pre {
          white-space: pre-wrap;
    }
  </style>
  '''))
get_ipython().events.register('pre_run_cell', set_css)

In [ ]:
from llama_index.core import SimpleDirectoryReader, ServiceContext, VectorStoreIndex
from llama_index.llms.openai import OpenAI

from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.core.query_engine import SubQuestionQueryEngine


In [ ]:
llm = OpenAI(temperature=0, model="gpt-3.5-turbo", max_tokens=1)
service_context = ServiceContext.from_defaults(llm=llm)

<ipython-input-5-c96836249eb9>:2: DeprecationWarning: Call to deprecated class method from_defaults. (ServiceContext is deprecated, please use `llama_index.settings.Settings` instead.) -- Deprecated since version 0.10.0.
  service_context = ServiceContext.from_defaults(llm=llm)


In [ ]:
medicare_2023 = SimpleDirectoryReader(input_files=["medicare_2023.pdf"]).load_data()
medicare_2024 = SimpleDirectoryReader(input_files=["medicare_2024.pdf"]).load_data()

In [ ]:
# setup baseline index
base_index = VectorStoreIndex.from_documents(medicare_2023 + medicare_2024)
base_engine = base_index.as_query_engine(similarity_top_k=4)

In [ ]:
response = base_engine.query("Name All the measures in 2023?")
print(str(response))

Topical Fluoride for Children (TFC), Oral Evaluation, Dental Services (OED), Deprescribing of Benzodiazepines in Older Adults (DBO), Emergency Department Visits for Hypoglycemia in Older Adults with Diabetes (EDH), Cervical Cancer Screening (CCS -E), Social Need Screening and Intervention (SNS -E).


In [ ]:
print(response.source_nodes[2].get_content())

6  
HEDIS MY 2023, Volume 2


In [ ]:
response = base_engine.query("Compare and contrast between 2023 and 2024")
print(str(response))

In 2024, there are no new measures introduced, while in 2023, specific measures like Colorectal Cancer Screening, Spirometry Testing for COPD, Follow-Up Care for Children with ADHD Medication, Metabolic Monitoring for Children on Antipsychotics, Non-Recommended Cervical Cancer Screening, Ambulatory Care, and Inpatient Utilization measures were retired. Additionally, the HbA1c Control for Patients With Diabetes measure was revised to Glycemic Status Assessment for Patients With Diabetes in 2024. Furthermore, in 2024, there were technical specification updates freezing the specifications on April 1, 2024, with the release of the HEDIS MY 2024 Volume 2 Technical Update memo and Value Set Directory, which organizations are accountable for.


In [ ]:
print(response.source_nodes[0].get_content(metadata_mode="all"))

page_label: 3
file_name: medicare_2024.pdf
file_path: medicare_2024.pdf
file_type: application/pdf
file_size: 2261137
creation_date: 2024-07-25
last_modified_date: 2024-07-25

Overview  3 
HEDIS MY 2024 , Volume 2  What’s New in Volume 2?  
New measures  • There are no new measures for HEDIS MY 2024.  
Retired measures  • Colorectal Cancer Screening  (COL)*.  
• Use of Spirometry Testing in the Assessment and Diagnosis of COPD 
(SPR) . 
• Follow -Up Care for Children Prescribed ADHD Medication (ADD) *. 
• Metabolic Monitoring for Children and Adolescents on Antipsychotics 
(APM)* . 
• Non-Recommended Cervical Cancer Screening in Adolescent Females 
(NCS) . 
• Ambulatory Care (AMB) . 
• Inpatient Utilization —General Hospital/Acute Care (IPU).  
*Only the COL-E, ADD -E and APM -E measure s will be reported .  
Revised measures  For specific revisions, refer to each measure’s Summary of Changes  or to 
Appendix 1 for a complete summary.  
• The former Hemoglobin A1c (HbA1c) Control for P

In [ ]:
medicare_2023_index = VectorStoreIndex.from_documents(medicare_2023)

In [ ]:
medicare_2024_index = VectorStoreIndex.from_documents(medicare_2024)

In [ ]:
medicare_2023_engine = medicare_2023_index.as_query_engine(similarity_top_k=2)

In [ ]:
medicare_2024_engine = medicare_2024_index.as_query_engine(similarity_top_k=2)

In [ ]:
query_engine_tools = [
    QueryEngineTool(
        query_engine=medicare_2023_engine,
        metadata=ToolMetadata(
            name="medicare_2023",
            description="Provides information about Tech Doc for year 2023",
        ),
    ),
    QueryEngineTool(
        query_engine=medicare_2024_engine,
        metadata=ToolMetadata(
            name="medicare_2024",
            description="Provides information about Tech Doc for year 2024",
        ),
    ),
]

s_engine = SubQuestionQueryEngine.from_defaults(query_engine_tools=query_engine_tools)

In [ ]:
# response = s_engine.query("what are the Top 3 Changes in Controlling Blooad Pressure Measure between 2023 and 2024?")
# print(str(response))

In [ ]:
query_str = "Name all the HEDIS Measures in 2023 and 2024 in a structured format as points seperately."
base_response = base_engine.query(query_str)
response = s_engine.query(query_str)
print(str(response))

Generated 2 sub questions.
[medicare_2023] Q: List all HEDIS Measures for 2023
[medicare_2024] Q: List all HEDIS Measures for 2024
[medicare_2023] A: Topical Fluoride for Children (TFC), Oral Evaluation, Dental Services (OED), Deprescribing of Benzodiazepines in Older Adults (DBO), Emergency Department Visits for Hypoglycemia in Older Adults with Diabetes (EDH), Cervical Cancer Screening (CCS -E), Social Need Screening and Intervention (SNS -E)
[medicare_2024] A: Colorectal Cancer Screening (COL), Use of Spirometry Testing in the Assessment and Diagnosis of COPD (SPR), Follow-Up Care for Children Prescribed ADHD Medication (ADD), Metabolic Monitoring for Children and Adolescents on Antipsychotics (APM), Non-Recommended Cervical Cancer Screening in Adolescent Females (NCS), Ambulatory Care (AMB), Inpatient Utilization —General Hospital/Acute Care (IPU), Hemoglobin A1c (HbA1c) Control for Patients With Diabetes (HBD), Osteoporosis Management in Women Who Had a Fracture.
- HEDIS Measures 